# Notebook 09 — Residual stream

Paired with `theory/09_residual_stream.md`. **Mode:** mixed.

## QHMPC

**Question.**
1. Telescoping decomposition: norm of cumulative perturbation vs. depth at init.
2. Per-component contributions: relative magnitudes of $a^{(\ell)}$ vs. $m^{(\ell)}$.
3. Skip-connection survival: linear probe of input token from final residual stream.
4. Logit-lens: next-token accuracy as function of depth.
5. Layer ablation: which layer matters most?

**Method.** Build a small 4-layer transformer, train on next-token prediction over a tiny synthetic language, instrument the residual stream.

**Prediction.** *Paste §9.7 predictions.*

**Confidence.** *[low/medium/high]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Setup — instrumented 4-layer transformer

We expose the per-layer residual streams and per-layer attention/FFN contributions for inspection.

In [ ]:
V = 32
d_model = 64
L = 4
h = 4
seq_len = 16

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, h, batch_first=True, bias=False)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, r, attn_mask):
        a, _ = self.attn(self.ln1(r), self.ln1(r), self.ln1(r), attn_mask=attn_mask, need_weights=False)
        m = self.ffn(self.ln2(r + a))                # for tracing, treat ffn input as post-attention r
        # but for the residual stream we want a, m separately:
        a_only = a
        m_only = self.ffn(self.ln2(r + a_only))
        out = r + a_only + m_only
        return out, a_only, m_only

class T4(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(V, d_model)
        self.pos = nn.Parameter(torch.randn(seq_len, d_model) * 0.02)
        self.blocks = nn.ModuleList([Block() for _ in range(L)])
        self.ln_out = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, V, bias=False)

    def forward(self, seq, return_trace=False):
        r = self.embed(seq) + self.pos
        causal = nn.Transformer.generate_square_subsequent_mask(seq_len, dtype=torch.float, device=r.device)
        trace = {'r': [r.detach().clone()], 'a': [], 'm': []}
        for blk in self.blocks:
            r, a, m = blk(r, attn_mask=causal)
            if return_trace:
                trace['r'].append(r.detach().clone()); trace['a'].append(a.detach().clone()); trace['m'].append(m.detach().clone())
        logits = self.unembed(self.ln_out(r))
        return (logits, trace) if return_trace else logits

# Toy task: predict-next on a synthetic alphabet sequence with simple bigram structure.
def make_batch(B):
    seq = torch.randint(0, V, (B, seq_len + 1))
    return seq[:, :-1], seq[:, 1:]

torch.manual_seed(0)
model = T4()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(800):
    seq, tgt = make_batch(32)
    logits = model(seq)
    loss = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        print(f'step {step}: loss = {loss.item():.4f}')
print(f'final loss: {loss.item():.4f}  (random baseline: log V = {np.log(V):.4f})')

## Experiment 1 — telescoping norm growth

In [ ]:
# Initialize a fresh 4-layer model — check norm growth at init.
torch.manual_seed(1)
init_model = T4()
seq, _ = make_batch(8)
with torch.no_grad():
    _, trace_init = init_model(seq, return_trace=True)

norms_init = [r.norm(dim=-1).mean().item() for r in trace_init['r']]
norms_trained = None
with torch.no_grad():
    _, trace_trained = model(seq, return_trace=True)
norms_trained = [r.norm(dim=-1).mean().item() for r in trace_trained['r']]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(L + 1), norms_init, 'o-', label='at init')
ax.plot(range(L + 1), norms_trained, 's-', label='trained')
ax.set_xlabel('depth (layer index)'); ax.set_ylabel('mean ||r^(L)||')
ax.set_title('Residual-stream norm vs. depth')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 1.** *Did norm grow monotonically? Trained vs. init growth pattern?*

## Experiment 2 — per-component contributions

In [ ]:
a_norms = [a.norm(dim=-1).mean().item() for a in trace_trained['a']]
m_norms = [m.norm(dim=-1).mean().item() for m in trace_trained['m']]

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(L)
ax.bar(x - 0.2, a_norms, 0.4, label='||attn contribution||')
ax.bar(x + 0.2, m_norms, 0.4, label='||ffn contribution||')
ax.set_xlabel('layer'); ax.set_ylabel('mean norm')
ax.set_title('Per-layer per-sub-layer contribution norms (trained)')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 2.** *Were attn and ffn contributions of similar magnitude? Did this vary across layers?*

## Experiment 3 — input-token survival via linear probe

In [ ]:
# Collect (final residual stream, input token) pairs.
X_data, y_data = [], []
model.eval()
with torch.no_grad():
    for _ in range(50):
        seq, _ = make_batch(32)
        _, tr = model(seq, return_trace=True)
        r_final = tr['r'][-1]                          # (B, n, d)
        X_data.append(r_final.reshape(-1, d_model)); y_data.append(seq.reshape(-1))
X = torch.cat(X_data); y = torch.cat(y_data)

# Linear probe: train logistic regression to predict input token from final residual stream.
probe = nn.Linear(d_model, V, bias=False)
opt2 = torch.optim.Adam(probe.parameters(), lr=1e-2)
for _ in range(500):
    logits = probe(X)
    loss = F.cross_entropy(logits, y)
    opt2.zero_grad(); loss.backward(); opt2.step()
with torch.no_grad():
    acc = (probe(X).argmax(-1) == y).float().mean().item()
print(f'Linear probe accuracy on input token from r^(L): {acc:.3f}   (chance = {1/V:.3f})')

**Sub-finding 3.** *Did the input token survive 4 layers of additive contributions? At what accuracy?*

## Experiment 4 — logit lens

In [ ]:
# Apply unembedding (with output LN) to each intermediate residual stream; compare to true next token.
logit_lens_acc = []
with torch.no_grad():
    seq, tgt = make_batch(200)
    _, tr = model(seq, return_trace=True)
    for ell in range(L + 1):
        r = tr['r'][ell]
        logits = model.unembed(model.ln_out(r))
        acc = (logits.argmax(-1) == tgt).float().mean().item()
        logit_lens_acc.append(acc)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(L + 1), logit_lens_acc, 'o-')
ax.set_xlabel('depth (logit lens applied at this layer)')
ax.set_ylabel('next-token accuracy')
ax.set_title('Logit-lens accuracy across depth')
plt.tight_layout(); plt.show()
for ell, a in enumerate(logit_lens_acc):
    print(f'depth {ell}: acc = {a:.3f}')

**Sub-finding 4.** *Did accuracy rise smoothly with depth, or jump? At which layer did the prediction become useful?*

## Experiment 5 — layer ablation

Zero out one block's contribution (set $a^{(\ell)} + m^{(\ell)}$ to zero) and report the resulting next-token accuracy.

In [ ]:
def ablate_layer(model, layer_idx, seq):
    """Forward pass that skips a chosen layer's contribution to the residual stream."""
    r = model.embed(seq) + model.pos
    causal = nn.Transformer.generate_square_subsequent_mask(seq_len, dtype=torch.float, device=r.device)
    for ell, blk in enumerate(model.blocks):
        r_new, a, m = blk(r, attn_mask=causal)
        if ell == layer_idx:
            r = r        # skip this layer's contribution
        else:
            r = r_new
    return model.unembed(model.ln_out(r))

with torch.no_grad():
    seq, tgt = make_batch(500)
    baseline = (model(seq).argmax(-1) == tgt).float().mean().item()
    print(f'baseline (no ablation): {baseline:.3f}')
    print()
    for ell in range(L):
        logits = ablate_layer(model, ell, seq)
        acc = (logits.argmax(-1) == tgt).float().mean().item()
        print(f'ablate layer {ell}: acc = {acc:.3f}    (Δ = {acc - baseline:+.3f})')

**Sub-finding 5.** *Which layer was most important by ablation? Did early layers matter as much as late layers?*

## Finding

*3–6 sentences. Did the residual-stream picture hold — did the embedding survive 4 layers, did logit lens work, did the contributions decompose cleanly? What does this experiment make you more confident about with respect to mechanistic interpretability methods (logit lens, linear probes, attribution)?*